# UK Government Legislation Summary Tool

## Web Scraper

Below is a method for scraping readable policy text from https://www.legislation.gov.uk/ simply find the link to a compatable policy document and provide it to the tool.

An example policy you can use is https://www.legislation.gov.uk/ukpga/2025/18/contents

### Code

In [ ]:
%pip install -r requirements.txt -q

In [ ]:
import feedparser
from bs4 import BeautifulSoup
import requests
import easygui
import logging
import re
import unittest
import time
import pandas as pd
import customtkinter as ctk

In [ ]:
class gov_uk_scraper:

    def input_url():

        # Prompt the user to enter the URL of the legislation's Table of Contents page
        
        myvar = easygui.enterbox("Please enter the URL to the Table of Contents page of the legislation you want to assess:", "UK Legislation NLP Comparison Tool")
        if myvar is None:
            return None

        # Parse the RSS feed of the legislation's Table of Contents page
        # Includes error handling to catch any exceptions that may occur during parsing
        
        else:
            try:
                feed = feedparser.parse(myvar + "/data.feed")
                # Check if the feed has entries and get the first entry's link
                url = feed.entries[0].link
                return url
            # If no entires are found call error handling function
            except Exception:
                return gov_uk_scraper.input_error("notfound")

    def input_error(error):

        # Recall input_url() if the user selects "Retry" from the error message box
        def retry_input(option):
            if option == "Retry": 
                # Respecting https://www.legislation.gov.uk/robots.txt which requests a 5 second delay between requests to the site
                start = time.time()
                while time.time() - start < 5:
                    easygui.msgbox("Please wait 5 seconds between requests", "UK Legislation NLP Comparison Tool")
                return gov_uk_scraper.input_url()
            return None

        # Display error when no url provided or no legislation found in data feed
        if error == "notfound":
            option = easygui.buttonbox("Unable to find a link to that legislation", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)
        # Display error when the url provided is not a valid legislation.gov.uk page
        elif error == "wrongsite":
            option = easygui.buttonbox("The URL provided does not appear to be a valid legislation.gov.uk page. Please check the URL and try again.", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)
        # Display error when the legislation is only available as a PDF and cannot be processed by the tool
        elif error == "pdfpage":
            option = easygui.buttonbox("The legislation provided is only available as a PDF and unforunately incompatible with this tool.", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)
        # Display error when the legislation does not have an Explanatory Memorandum and cannot be compared
        elif error == "noemlink":
            option = easygui.buttonbox("The legislation provided does not appear to have an Explanatory Memorandum. No Comparison can be made", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)

    def get_legislation_text(url):
        # Calls error handling function if legislation is only available as PDF
        if url.endswith(".pdf"):
            return gov_uk_scraper.input_error("pdfpage")
        
        #Initial web scraping
        else:
            response = requests.get(url)
            soup = BeautifulSoup(response.content, 'html.parser')

            # Final validation check to ensure website is legislation.gov.uk
            site_check = "legislation.gov.uk" in soup.find("div", id="header").get_text()
            # Checking for the presence of an Explanatory Memorandum link on the legislation page
            em_check = soup.find("li", id=["legEmLink","legEnLink"])
            if not site_check:
                return gov_uk_scraper.input_error("wrongsite")
            elif not em_check:
                return gov_uk_scraper.input_error("noemlink")
            # Text of legislation is contained within a div with class "LegSnippet"
            else:
                legislation_text = soup.find("div", class_="LegSnippet")
                return legislation_text

    def prettify_text(content):
        lines = []
        for el in content.find_all(["h1", "h2", "h3", "h4", "p", "li", "td", "div"]):
            # skip elements that just contain other elements we'll already visit —
            # otherwise you'd get duplicated text
            if el.find(["p", "li", "td", "div", "h1", "h2", "h3", "h4"]):
                continue
            text = el.get_text(" ", strip=True)
            if text:
                lines.append(text)
        readable = "\n\n".join(lines)
        return readable

    def main():
        url = gov_uk_scraper.input_url()
        # Exiting Program if no valid URL is provided
        if url is None:
            return "No valid URL provided. Exiting program."
        else:
            legislation_text = gov_uk_scraper.get_legislation_text(url)
            readable_text = gov_uk_scraper.prettify_text(legislation_text)
            print(readable_text)
            return readable_text

In [ ]:
text = gov_uk_scraper.main()

## API Method

Below is a method for collecting the legislation wording from https://www.legislation.gov.uk/ simply find the link to a compatable policy document and provide it to the tool.

WIP: adding the ability to search for legislation using key words.

An example policy you can use is https://www.legislation.gov.uk/ukpga/2025/18/contents

### GUI Code and Functions

In [7]:
import customtkinter as ctk
from bs4 import BeautifulSoup
import requests
import feedparser
import pandas as pd
import re

class App(ctk.CTk):

    def __init__(self):
        super().__init__()

        self.configure(fg_color="#23272D")
        self.title("UK Legislation Summarisation Tool")
        self.geometry("700x275")

        self.inptdata = ctk.CTkEntry(master=self, placeholder_text="Enter Search Terms", width=620, height=60)
        self.inptdata.configure(fg_color="#343739", text_color="#fff", corner_radius=5, border_width=2, border_color="#505557", font=ctk.CTkFont(family="Helvetica", size=20))
        self.inptdata.place(x=40, y=100)
        self.inptdata.bind("<Return>", lambda event: self._on_search())

        # Use `command=` on CTkButton instead of bind
        self.searchbtn = ctk.CTkButton(master=self, text="Search!", command=self._on_search, width=200, height=60)
        self.searchbtn.configure(fg_color="#dc8000", hover_color="#ecad55", text_color="#fff", corner_radius=5, font=ctk.CTkFont(family="Helvetica", size=32))
        self.searchbtn.place(x=140, y=180)

        self.radiotype_var = ctk.IntVar()

        self.radiotype_0 = ctk.CTkRadioButton(master=self, variable=self.radiotype_var, text="Primary Legislation", value=0, width=200, height=60)
        self.radiotype_0.configure(fg_color="#dc8000", hover_color="#ecad55", text_color="#fff", corner_radius=5, border_color="#8e5b14", font=ctk.CTkFont(family="Helvetica"))
        self.radiotype_0.place(x=360, y=160)

        self.radiotype_1 = ctk.CTkRadioButton(master=self, variable=self.radiotype_var, text="Secondary Legislation", value=1, width=200, height=60)
        self.radiotype_1.configure(fg_color="#dc8000", hover_color="#ecad55", text_color="#fff", corner_radius=5, border_color="#8e5b14", font=ctk.CTkFont(family="Helvetica"))
        self.radiotype_1.place(x=360, y=200)
        self.radiotype_var.set(0)

        self.lbldata = ctk.CTkLabel(master=self, text="Enter the URL of the legislation you'd like to summarise below, Alternatively enter a search term e.g. Data Protection Act to be given a list of options", width=620, height=60, wraplength=600)
        self.lbldata.configure(fg_color="transparent", text_color="#fff", font=ctk.CTkFont(family="Helvetica", size=18))
        self.lbldata.place(x=40, y=20)

    def _show_results(self, entries):
        results_window = ctk.CTkToplevel(self)
        results_window.title("Legislation Search Results")
        results_window.configure(fg_color="#23272D")
        results_window.geometry("800x600")
        results_window.transient(self)

        heading = ctk.CTkLabel(results_window, text="Select legislation", text_color="#fff", font=ctk.CTkFont(family="Helvetica", size=24, weight="bold"))
        heading.pack(pady=(20, 10))

        results_frame = ctk.CTkScrollableFrame(results_window, width=720, height=430, fg_color="#343739")
        results_frame.pack(padx=20, pady=10, fill="both", expand=True)

        selected_url = ctk.StringVar(value="")

        for entry in entries:
            result_frame = ctk.CTkFrame(results_frame, fg_color="#414549")
            result_frame.pack(fill="x", padx=10, pady=8)

            radio_button = ctk.CTkRadioButton(result_frame, text=entry.title, variable=selected_url, value=entry.link, text_color="#fff", fg_color="#dc8000", hover_color="#ecad55", font=ctk.CTkFont(family="Helvetica", size=16))
            radio_button.pack(anchor="w", padx=12, pady=(10, 4))

            summary = getattr(entry, "summary", "No summary available.")
            summary_label = ctk.CTkLabel(result_frame, text=summary, text_color="#d7dadd", justify="left", anchor="w", wraplength=680)
            summary_label.pack(fill="x", padx=38, pady=(0, 4))

            link_label = ctk.CTkLabel(result_frame, text=entry.link, text_color="#ecad55", justify="left", anchor="w", wraplength=680)
            link_label.pack(fill="x", padx=38, pady=(0, 10))

        def use_selected_result():
            if selected_url.get():
                self.inptdata.delete(0, "end")
                self.inptdata.insert(0, selected_url.get())
                self._on_search()
                results_window.destroy()

        select_button = ctk.CTkButton(results_window, text="Use selected legislation", command=use_selected_result, width=240, height=45)
        select_button.configure(fg_color="#dc8000", hover_color="#ecad55", text_color="#fff", corner_radius=5, font=ctk.CTkFont(family="Helvetica", size=32))
        select_button.pack(pady=(5, 20))

    def data_xml_url(self, url):
        match = re.match(r"^(https?://www\.legislation\.gov\.uk/[a-z]+/\d{4}/\d+)(?:/.*)?/?$", url)
        if not match:
            raise ValueError("Invalid legislation.gov.uk URL")
        return match.group(1) + "/data.xml"

    def process_legislation(self, xml_url):
        url = xml_url
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, "lxml-xml")
        readable_text = soup.get_text("\n\n", strip=True)
        print(readable_text)

    def _on_search(self):
        query = self.inptdata.get().strip()
        self.inptdata.delete(0, "end")
        if not query:
            print("Enter search terms")
            return

        if re.search(r"https?://www\.legislation\.gov\.uk/[a-z]+/\d{4}/\d+", query):
            data_xml_url = self.data_xml_url(query)
            print(self.process_legislation(data_xml_url))
            return

        selected = self.radiotype_var.get()
        kind_map = {0: "primary", 1: "secondary"}
        kind = kind_map.get(selected, "unknown")

        url = f"https://www.legislation.gov.uk/search?results-count=10&title={query}&type={kind}"
        headers = {"Accept": "application/atom+xml"}
        resp = requests.get(url, headers=headers, timeout=15)
        feed = feedparser.parse(resp.content)
        if not feed.entries:
            print("No results found for query:", query, "Kind:", kind)
            return

        self._show_results(feed.entries)

if __name__ == "__main__":
    app = App()
    app.mainloop()


http://www.legislation.gov.uk/ukpga/2018/12

Data Protection Act 2018

An Act to make provision for the regulation of the processing of information relating to individuals; to make provision in connection with the Information Commissioner's functions under certain regulations relating to information; to make provision for a direct marketing code of practice; and for connected purposes.

text

text/xml

en

Statute Law Database

2026-07-09

Expert Participation

2026-06-19

Data Protection Act 2018

s. 205(2)(l)

Data (Use and Access) Act 2025

s. 117(4)(a)

s. 142(1)

Data Protection Act 2018

s. 3(8)

Data (Use and Access) Act 2025

s. 118(3)

s. 142(1)

Data Protection Act 2018

s. 114 and cross-heading

Data (Use and Access) Act 2025

s. 118(4)

s. 142(1)

Data Protection Act 2018

s. 206

Table

Data (Use and Access) Act 2025

s. 118(5)

s. 142(1)

Data Protection Act 2018

s. 214(1)(a)

Data (Use and Access) Act 2025

s. 118(6)(a)

s. 142(1)

Data Protection Act 2018

s. 214(1)(b)

In [ ]:
def data_xml_url(url):
    match = re.match(r"^(https?://www\.legislation\.gov\.uk/[a-z]+/\d{4}/\d+)(?:/.*)?/?$", url)
    if not match:
        raise ValueError("Invalid legislation.gov.uk URL")
    return match.group(1) + "/data.xml"

# All of these produce the same /data.xml URL
urls = [
    "http://www.legislation.gov.uk/ukpga/2018/12/contents/2026-06-19/",
    "https://www.legislation.gov.uk/ukpga/2018/12/2026-06-19/",
    "https://www.legislation.gov.uk/ukpga/2018/12/",
    "https://www.legislation.gov.uk/ukpga/2018/12/contents/",
]

url = data_xml_url(urls[0])
print(url)

response = requests.get(url, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.content, "lxml-xml")
readable_text = soup.get_text("\n\n", strip=True)
print(readable_text)

## Summarisation Module Ideation

### LSA Exploration 

In [ ]:
import requests # Import requests library

from bs4 import BeautifulSoup # Add BeautifulSoup for HTML parsing

from sumy.parsers.plaintext import PlaintextParser

from sumy.nlp.tokenizers import Tokenizer

from sumy.summarizers.luhn import LuhnSummarizer # Import LuhnSummarizer

from sumy.summarizers.lex_rank import LexRankSummarizer # Import LexRankSummarizer

from sumy.summarizers.lsa import LsaSummarizer # Import LsaSummarizer

from sumy.nlp.stemmers import Stemmer

from sumy.utils import get_stop_words

import nltk 

nltk.download('punkt_tab')

In [ ]:
# LSA Extractive Summarization Example



def lsa_summarize(input_data, sentence_count=2, input_type="text"):


#    Summarize text using the LSA algorithm.



 #   Args:

 #       input_data (str): The input text or URL to summarize.
#
 #       sentence_count (int): Number of sentences for the summary.
#
 #       input_type (str): Type of input - “text” or “url”.
#


 #   Returns: list: Summary sentences.

    if input_type == "url":

        response = requests.get(input_data)

        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser') # Parse HTML content

        text = soup.get_text(separator=' ') # Extract plain text

    else: text = input_data



    # Parse the input text

    parser = PlaintextParser.from_string(text, Tokenizer("english"))



    # Initialize LSA summarizer with stemmer

    summarizer = LsaSummarizer(Stemmer("english"))

    summarizer.stop_words = get_stop_words("english")



    # Generate summary

    summary = summarizer(parser.document, sentence_count)

    return summary



# Test with sample text

sample_text = """

Text summarization is an important area of natural language processing (NLP) that focuses on condensing large amounts of text into shorter, coherent summaries. Modern approaches can identify the main ideas in a document and present them with minimal human involvement. Extractive methods select representative sentences directly from the source text, while abstractive methods generate new phrasing based on the original meaning. These techniques are increasingly used in information retrieval, research analysis, and other applications where quick understanding of text is essential.

"""



# Summarize plain text

summary = lsa_summarize(readable_text, 5, input_type="text")

print("Summary from text:")

for sentence in summary:

    print(sentence)

### Transformer Abstractive Summarization Exploration

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration
import torch

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

def generate_summary(text):
    inputs = tokenizer.encode("summarize: " + text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(inputs, max_length=150, min_length=50, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [ ]:
#article = "Your sample article text goes here..."
summary = generate_summary(readable_text)
#print("Original Text:", readable_text)
print("Summary:", summary)

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.legislation.gov.uk/title/data"
headers = {"Accept": "application/atom+xml"}

resp = requests.get(url, headers=headers, timeout=15)
resp.raise_for_status()

# raw XML text
#xml_text = resp.text
#print(xml_text[:1000])

# optional: parse Atom with BeautifulSoup (xml mode)
soup = BeautifulSoup(resp.content, "lxml-xml")
entries = soup.find_all("entry")
entries
#print(f"Found {len(entries)} entries")

In [ ]:
import requests
from bs4 import BeautifulSoup
import json

def fetch_legislation_via_api(url_or_path):
    """Fetch legislation text using legislation.gov.uk data endpoints.
    Accepts a full URL or a path like 'ukpga/2025/18'. Returns plain text or raises."""
    if url_or_path.startswith('http'):
        base = url_or_path.rstrip('/')
    else:
        base = 'https://www.legislation.gov.uk/' + url_or_path.strip('/')

    candidates = [base + '/data.json', base + '/data.xml', base + '/data']
    for endpoint in candidates:
        try:
            resp = requests.get(endpoint, timeout=15)
            if resp.status_code != 200:
                continue
            ct = resp.headers.get('Content-Type','')
            if 'application/json' in ct or endpoint.endswith('.json'):
                payload = resp.json()
                texts = []
                def collect_strings(obj):
                    if isinstance(obj, str):
                        texts.append(obj)
                    elif isinstance(obj, list):
                        for v in obj: collect_strings(v)
                    elif isinstance(obj, dict):
                        for v in obj.values(): collect_strings(v)
                collect_strings(payload)
                return '\n\n'.join([t.strip() for t in texts if t.strip()])
            else:
                # treat as XML/HTML
                soup = BeautifulSoup(resp.content, 'lxml-xml')
                text = soup.get_text('\n\n', strip=True)
                if text:
                    return text
        except Exception:
            continue
    raise RuntimeError('Could not fetch legislation via API endpoints.')

In [ ]:
from transformers import LEDTokenizer, LEDForConditionalGeneration
import torch

model_name = "allenai/led-large-16384-arxiv"
tokenizer = LEDTokenizer.from_pretrained(model_name)
model = LEDForConditionalGeneration.from_pretrained(model_name)

def generate_summary(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=16384,
        truncation=True,
    )

    global_attention_mask = torch.zeros_like(inputs["input_ids"])
    global_attention_mask[:, 0] = 1

    summary_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        global_attention_mask=global_attention_mask,
        max_length=256,
        min_length=30,
        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        early_stopping=True,
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
summary = generate_summary(readable_text)
print("Summary:", summary)

### Rouge scoring to identify model accuracy

In [ ]:
from rouge_score import rouge_scorer

reference_summary = """
An Act to make provision for the regulation of the processing of information relating to individuals; to make provision in connection with the Information Commissioner’s functions under certain regulations relating to information; to make provision for a direct marketing code of practice; and for connected purposes.
"""

generated_summary = generate_summary(readable_text)

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

scores = scorer.score(reference_summary, generated_summary)

for metric, score in scores.items():
    print(f"{metric}:")
    print(f"  Precision: {score.precision:.3f}")
    print(f"  Recall:    {score.recall:.3f}")
    print(f"  F1:        {score.fmeasure:.3f}")

### Unit Testing

In [ ]:
# To be included before Summative!

# Early Testing Cells

Easy Gui Exploration

In [ ]:
myvar = easygui.enterbox("test", "test")
print(myvar)

Initial Functions for Web Scraper

In [ ]:
def get_legislation_text(url):
    if url.endswith(".pdf"):
        return input_error("pdfpage")
    else:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        #Final validation check to ensure website is legislation.gov.uk
        site_check = "legislation.gov.uk" in soup.find("div", id="header").get_text()
        em_check = soup.find("li", id="legEmLink")
        if not site_check:
            return input_error("wrongsite")
        elif not em_check:
            return input_error("noemlink")
        else:
            legislation_text = soup.find("div", class_="LegSnippet")
        return legislation_text

In [ ]:
def prettify_text(content):
    lines = []
    for el in content.find_all(["h1", "h2", "h3", "h4", "p", "li", "td", "div"]):
        # skip elements that just contain other elements we'll already visit —
        # otherwise you'd get duplicated text
        if el.find(["p", "li", "td", "div", "h1", "h2", "h3", "h4"]):
            continue
        text = el.get_text(" ", strip=True)
        if text:
            lines.append(text)
    readable = "\n\n".join(lines)
    return readable

In [ ]:
def main():
    url = input_url()
    legislation_text = get_legislation_text(url)
    readable_text = prettify_text(legislation_text)
    print(readable_text)

In [ ]:
main()

Exploring Legislation OpenAPI Search Module

In [ ]:
from bs4 import BeautifulSoup
import requests

url = "https://www.legislation.gov.uk/search?results-count=10&title=Data Protection Act&type=primary"
headers = {"Accept": "application/atom+xml"}

resp = requests.get(url, headers=headers, timeout=15)
 
feed = feedparser.parse(resp.content)
#print(len(feed.entries))
#print(feed.entries[0].title)
#print(feed.entries[0].link)
#print(feed.entries[0])

content = requests.get(feed.entries[0].link)
soup = BeautifulSoup(content.content, 'html.parser')
print(feed.entries[0].link)
print(soup.prettify())

'''             
legs =[]
for leg in feed.entries:
    legs.append((leg.title, leg.link, leg.summary, leg.updated))

df = pd.DataFrame(legs, columns=["Title", "Link", "Summary", "Updated"])

df.head(20)'''